# ARC-AGI-3 — final solution notebook

Our final notebook for the [ARC Prize 2026 – ARC-AGI-3](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3)
Kaggle competition. One notebook for Kaggle (test runs and the real submission) and Colab (GPU test runs).

**Built on the work of others.** This notebook was developed on the basis of Daniel Franzen's ARC-AGI-3 Milestone 2 solution,
which itself builds on Tufa Labs' Duck harness and on the serving work of several community members.
Please read *Credits and prior work* below before anything else; most of what makes this notebook run is theirs.

Model: Qwen3.8-Flash-Next W4A16 (Intel AutoRound) with the albucino MTP drafter, served by SGLang
from the Pennyroyal wheelhouse. Agent: the ARC3-Inference tool agent (Tufa Labs' Duck) on the TAAF framework.

**Where the code lives.** The agent and every helper this notebook calls come from one directory,
`harness/` in the repository (`HARNESS_MANIFEST.json`, `ARC3-Inference/`, `notebook/`, `tools/`).
Upload that directory as a Kaggle dataset and to Google Drive, and point `HARNESS_DIR_KAGGLE` /
`HARNESS_DIR_COLAB` below at the uploaded copies. The TAAF source bundle is still read from its own
dataset; the harness replaces its `src/ARC3-Inference` tree. There are no patch files.

**How the cells run.** On Kaggle every cell runs in the notebook kernel. On Colab the bootstrap cell
starts a pinned Python 3.12 harness process and forwards every later cell to it, so the cells are
identical on both. A cell whose first line is `# arc3-step: parent` stays in the Colab kernel; the
other `# arc3-step:` lines tell the Colab adapter which cells need extra work (model start, run, export).

Run all cells in order in a fresh session. Edit only the configuration cell and, for experiments,
the runtime settings cell.


## Credits and prior work

This notebook stands on the shoulders of people who published their work openly during the competition.
We are sincerely grateful to all of them. Thank you for sharing.

**Daniel Franzen — ARC-AGI-3 Milestone 2 solution.**
Our notebook was developed on the basis of his
[competition notebook](https://www.kaggle.com/code/dfranzen/arc-agi-3-milestone-2-solution)
([source and write-up](https://github.com/da-fr/arc-agi-3-solution), Apache-2.0). The model choice, the optimized
SGLang serving setup, the priority scheduler, context management, the perception improvements (board-difference,
death and animation images), executable undo, retained Python functions and the stale-state / no-op guards are
his work, and every result we report was obtained on top of them. The Kaggle inputs this notebook attaches are his
published copies: the Pennyroyal wheelhouse (`dfranzen/pennyroyal-v253`), the TAAF source bundle
(`dfranzen/taaf-kaggle-source-bundle-copy`), the Intel AutoRound W4A16 Qwen3.8-Flash-Next checkpoint and the
albucino Qwen3.8-Flash-Next MTP drafter. Our sampling settings note where we moved away from his values.

**Jeroen Cottaar and Tufa Labs — The Duck harness and TAAF.**
The tool-using agent (`ARC3-Inference`, "The Duck") and the game-execution framework (TAAF) come from Tufa Labs'
[Duck harness](https://github.com/Tufalabs/duck-harness), the winning solution of ARC-AGI-3 Milestone 1, described
in their [technical write-up](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/discussion/717133).
Franzen's fork, and therefore ours, is based on upstream commit `7652836`.

**Serving.** The offline SGLang runtime builds on the work of
[John Pezzulli (Pennyroyal)](https://github.com/jpezzulli/sglang-rtxpro6000),
[Mamy Ratsimbazafy](https://github.com/mratsim/sglang-qwen38fn-sm120-turbo) and
[Gabriel Olympie](https://github.com/gabrielolympie/sglang-flashnext-sm120), and on the
[SGLang](https://github.com/sgl-project/sglang) project.

**Models.** The Qwen team for Qwen3.8-Flash-Next, Intel for the AutoRound quantization, and the authors of the
albucino MTP drafter.

**ARC Prize Foundation** for ARC-AGI-3, its public games and this competition.

### What this repository adds

On top of that base we kept one notebook for Kaggle and Colab and consolidated the harness into source
(no patch files). The agent-side additions are: Action Evidence v2 (truthful action results and guard messages),
a unified prompt contract (single system prompt, truncation recovery, stall note, time cues), an exploration note,
a goal check over the model's own `goal_violations(frame)`, a simulator contract (`observe` / `step` checked against
the level's transitions and guarding action batches), death context on game-over turns, and context compaction
instead of silent truncation. We also tuned the scheduler's token scale, added a Colab runtime, and record a
manifest, the exact system prompt and file digests with every run. The repository README describes these changes
and our experiments, including the ones that did not work.

Where our code differs from Franzen's, mistakes are ours, not his.


In [ ]:
# arc3-step: parent
# ---------------------------------------------------------------- run configuration
RUNTIME_ENVIRONMENT = "kaggle"   # "kaggle" | "colab"
RUN_STAGE = "PILOT"              # PILOT: pilot games, 45 min each | FULL: 25 public games, 120 min each
PILOT_GAME_PREFIXES = ["sk48", "bp35", "lf52", "ar25", "cd82", "ft09", "lp85", "sb26", "r11l", "vc33"]
PUBLIC_EXCLUDED_GAMES = []
RUN_CPU_SELF_TESTS = True
COLAB_DISCONNECT_AFTER_EXPORT = True   # Colab: release the GPU after the bundle reaches Drive

# Harness directory (harness/ in the repository). Set these to the uploaded copies.
HARNESS_DIR_KAGGLE = "/kaggle/input/datasets/ahoirg/arc3-harness"   # searched under /kaggle/input if absent
HARNESS_DIR_COLAB = "/content/drive/MyDrive/ARC3/inputs/harness"

# Kaggle inputs (attached in kernel-metadata.json) and outputs.
KAGGLE_INPUTS = {
    "wheelhouse": "/kaggle/input/datasets/dfranzen/pennyroyal-v253",
    "model": "/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1",
    "draft_model": "/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1",
    "source_bundle": "/kaggle/input/datasets/dfranzen/taaf-kaggle-source-bundle-copy",
    "competition": "/kaggle/input/competitions/arc-prize-2026-arc-agi-3",
}
KAGGLE_OUTPUT_DIR = "/kaggle/working"
KAGGLE_BUNDLE_DIR = "/kaggle/taaf-kaggle-source-share"

# Colab inputs come from this Drive JSON (model manifest, dataset copies, output folders).
COLAB_CONFIG_PATH = "/content/drive/MyDrive/ARC3/colab_config.json"

assert RUNTIME_ENVIRONMENT in {"kaggle", "colab"}, 'RUNTIME_ENVIRONMENT must be "kaggle" or "colab"'
assert RUN_STAGE in {"PILOT", "FULL"}
assert PILOT_GAME_PREFIXES and len(set(PILOT_GAME_PREFIXES)) == len(PILOT_GAME_PREFIXES)
PUBLIC_RUNTIME_MINUTES = 45 if RUN_STAGE == "PILOT" else 120   # 45 min ~ average decode time per game in a submission

In [ ]:
# arc3-step: parent
# ---------------------------------------------------------------- bootstrap: locate the harness
import sys
import zipfile
from pathlib import Path


def _arc3_locate_harness(configured, search_roots=()):
    """Find the directory holding HARNESS_MANIFEST.json (a dataset may wrap it in folders or a zip)."""
    marker = "HARNESS_MANIFEST.json"
    candidates = [Path(configured)] if configured else []
    for candidate in candidates:
        if candidate.is_file() and candidate.suffix == ".zip":
            target = Path("/tmp/arc3-harness")
            with zipfile.ZipFile(candidate) as archive:
                archive.extractall(target)
            candidate = target
        if (candidate / marker).is_file():
            return candidate
        if candidate.is_dir():
            found = sorted(candidate.glob(f"*/{marker}")) + sorted(candidate.glob(f"*/*/{marker}"))
            if found:
                return found[0].parent
    for root in map(Path, search_roots):
        found = sorted(p for depth in range(1, 6) for p in root.glob("/".join(["*"] * depth) + f"/{marker}"))
        if found:
            print(f"Harness not at {configured!r}; using {found[0].parent}", flush=True)
            return found[0].parent
    raise FileNotFoundError(
        f"Harness directory not found at {configured!r}. Upload harness/ and set "
        "HARNESS_DIR_KAGGLE / HARNESS_DIR_COLAB in the configuration cell.")


if RUNTIME_ENVIRONMENT == "colab":
    from google.colab import auth, drive
    auth.authenticate_user()
    drive.mount("/content/drive")
    HARNESS_DIR = _arc3_locate_harness(HARNESS_DIR_COLAB)
else:
    HARNESS_DIR = _arc3_locate_harness(HARNESS_DIR_KAGGLE, search_roots=["/kaggle/input"])
NOTEBOOK_SUPPORT_DIR = HARNESS_DIR / "notebook"
if str(NOTEBOOK_SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_SUPPORT_DIR))
import arc3nb

HARNESS_INFO = arc3nb.verify_harness(HARNESS_DIR)
ARC3_BUNDLE_NAME = f"ARC3_{RUN_STAGE}_{HARNESS_INFO['version']}_run_bundle.zip"
ARC3_SHARED_SETTINGS = ("RUNTIME_ENVIRONMENT", "RUN_STAGE", "PILOT_GAME_PREFIXES", "PUBLIC_EXCLUDED_GAMES",
                        "PUBLIC_RUNTIME_MINUTES", "RUN_CPU_SELF_TESTS", "HARNESS_INFO")
print(f"Harness {HARNESS_INFO['version']} at {HARNESS_DIR} (tree {HARNESS_INFO['tree_sha256'][:12]})", flush=True)

if RUNTIME_ENVIRONMENT == "kaggle":
    RUNTIME_PATHS = dict(KAGGLE_INPUTS, harness=str(HARNESS_DIR), working=KAGGLE_OUTPUT_DIR, bundle=KAGGLE_BUNDLE_DIR)
else:
    _arc3_adapter = NOTEBOOK_SUPPORT_DIR / "colab_adapter.py"
    exec(compile(_arc3_adapter.read_text(encoding="utf-8"), str(_arc3_adapter), "exec"), globals())

## Environment and harness install

Copies the TAAF source bundle, replaces its `src/ARC3-Inference` with the harness tree and compiles it.

In [ ]:
# arc3-step: setup
import json
import os
import pickle
import subprocess
import sys
import threading
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

if str(NOTEBOOK_SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_SUPPORT_DIR))
import arc3nb

if "inference.agent.tool_agent" in sys.modules:
    raise RuntimeError("The agent is already imported; restart the session and run all cells in order.")

# Kaggle may cancel a session that shows no activity, so the harness starts even while the model
# server is still loading (the first game performs one warm-up RESET); SERVER_STARTUP_TIMEOUT bounds that.
NOTEBOOK_START_TIME = NOTEBOOK_START_EPOCH = time.time()
SERVER_STARTUP_TIMEOUT = 12 * 60
# True only inside a real competition rerun; switches diagnostics and the soft deadline.
TRUE_SUBMISSION = (RUNTIME_ENVIRONMENT == "kaggle"
                   and os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"})

WHEELHOUSE_DIR = RUNTIME_PATHS["wheelhouse"]
MODEL_DIR = RUNTIME_PATHS["model"]
DRAFT_MODEL_DIR = RUNTIME_PATHS["draft_model"]
WORKING_DIR = Path(RUNTIME_PATHS["working"])
WORKING_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR = Path(RUNTIME_PATHS["bundle"])
HARNESS_INSTALL = arc3nb.install_harness(RUNTIME_PATHS["source_bundle"], HARNESS_DIR, BUNDLE_DIR)

SERVED_MODEL_NAME = "flashnext"
SERVED_MODEL_HOST = "127.0.0.1"
SERVED_MODEL_PORT = 8001
SERVER_BASE_URL = f"http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/v1"
USE_PRIORITY_SCHEDULING = True
print(f"runtime={RUNTIME_ENVIRONMENT} TRUE_SUBMISSION={TRUE_SUBMISSION} working={WORKING_DIR}", flush=True)

## Runtime settings

Everything the agent, scheduler and framework read from the environment. The agent reads these when it is
first imported, so change them here (before the next cells) and restart the session to apply a change.

In [ ]:
# arc3-step: settings
setup_env = {
    # ---- run mode
    "MPLBACKEND": "Agg",
    "TAAF_RUN_AS_SUBMISSION": "1" if TRUE_SUBMISSION else "0",
    "TAAF_MINIMAL_DIAGNOSTICS": "1" if TRUE_SUBMISSION else "0",   # no periodic JSON/HTML diagnostics in submission
    "ONLY_RESET_LEVELS": "true",                                    # RESET restarts the level, not the game
    "ARC3_WARMUP_ACTION_GAMES": "1",                                # first game sends one RESET while the server loads
    "ARC3_HTTP_RETRY_INITIAL_SECONDS": "900",                       # tolerate a model server that is still starting
    "USE_TF": "0", "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_TORCHVISION": "1",

    # ---- model endpoint
    "LOCAL_ANALYZER_BASE_URL": SERVER_BASE_URL, "OPENAI_BASE_URL": SERVER_BASE_URL,
    "LOCAL_ANALYZER_PROVIDER": "vllm", "OPENAI_PROVIDER": "vllm",
    "LOCAL_ANALYZER_MODEL_ID": SERVED_MODEL_NAME, "INFERENCE_ANALYZER_MODEL": SERVED_MODEL_NAME,
    "LOCAL_ANALYZER_APP_NAME": "ARC3 Agent Harness",

    # ---- sampling and turns
    "ARC3_REASONING_HISTORY_KEY": "reasoning_content",
    "LOCAL_ANALYZER_ENABLE_THINKING": "true",
    "LOCAL_ANALYZER_TEMPERATURE": "1.0", "LOCAL_ANALYZER_TOP_P": "0.95", "LOCAL_ANALYZER_TOP_K": "20",   # r12: the Qwen3.8 card's thinking-mode sampling (was 0.7, Daniel Franzen's value)
    "LOCAL_ANALYZER_TOOL_STEPS": "0",           # no cap on tool calls per turn
    "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",        # seconds per python call
    "LOCAL_ANALYZER_YIELD_SECONDS": "0",
    "LOCAL_ANALYZER_YIELD_TOKENS": "0",         # no mid-turn yields: a yield re-sent the whole opener
    "ARC3_TRUNCATION_RECOVERY": "1",            # cut-off reply -> short "state hypothesis, then call" follow-up
    "ARC3_STALL_SECONDS": "150",                # note after this long on one turn without an action
    "ARC3_TIME_CUES": "1",                      # remaining game time and level spend in each opener
    "ARC3_EXPLORATION_NOTE": "1",               # opener lists untried actions/objects and state-specific no-ops
    "ARC3_EXPLORATION_MIN_ACTIONS": "6",        # ...once this many actions were spent on the level
    "ARC3_GOAL_CHECK": "1",                     # run the model's retained goal_violations(frame) and report it
    "ARC3_GOAL_CHECK_NUDGE_ACTIONS": "8",       # ...or ask for one once this many actions were spent on the level
    "ARC3_SIM_CHECK": "1",                      # r15: replay the model's observe(frame) + step(state, action) over the level's transitions each turn; search(goal) plans over the same pair
    "ARC3_SIM_GUARD": "1",                      # r15: a batch stops at the first action the pair mispredicts (stop_reason sim_mismatch)
    "ARC3_SIM_CHECK_NUDGE_ACTIONS": "8",        # ...or say once per level that the pair is missing, after this many actions

    # ---- context
    "LOCAL_ANALYZER_CONTEXT_WINDOW": (116 + 12) * 1024,
    "LOCAL_ANALYZER_MAX_OUTPUT": 12 * 1024,   # r12: back from 8k; the 8k cap raised truncated text from 3-5% to 9-14% (r10)
    "ARC3_CONTEXT_DRAIN_TOKENS": 58 * 1024,
    "ARC3_HISTORY_ASSISTANT_TURNS": 150,
    "ARC3_HISTORY_TURN_DRAIN": 30,
    "ARC3_HISTORY_DRAIN_COALESCE": "1",
    "ARC3_CONTEXT_COMPACTION": "1",             # r17: before a drain removes old turns, the model writes notes that replace them (compaction, not silent truncation)
    "ARC3_COMPACTION_MARGIN_TOKENS": 16 * 1024, # ...triggered this far below the context budget, so the notes request fits the server context
    "ARC3_COMPACTION_MAX_TOKENS": 12 * 1024,    # r18: notes request uses the turn settings (tools, auto, thinking on) so the cached prefix is reused; cap includes thinking

    # ---- tool output and action results
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "3072",
    "ARC3_MIDDLE_TRUNCATION": "1",
    "ARC3_ACTION_ECHO": "1",
    "ARC3_ACTION_EVIDENCE": "1",
    "ARC3_OBSERVATION_DIR": str(WORKING_DIR / "observations"),   # passive per-action archive, never shown to the model

    # ---- vision and animation
    "MULTIMODAL_CONTEXT": "current_grid", "MULTIMODAL_UPSCALE": "10",
    "ARC3_DIFF_IMAGE": "1", "ARC3_GAMEOVER_DIFF_IMAGE": "1", "ARC3_ANIMATION": "1",
    "ARC3_DEATH_CONTEXT": "1",                 # r14: on a death turn, the fatal animation's scroll, object tracks and what the vanished object touched

    # ---- guards
    "ARC3_GUARDS_FROM_LEVEL": "1",
    "ARC3_BATCH_NOOP_BLOCK": "1",
    "ARC3_STALE_STATE_BLOCK": "1",

    # ---- persistent python functions
    "ARC3_PERSISTENT_FUNCTIONS": "1", "ARC3_PERSISTENT_FUNCTIONS_SCOPE": "game", "ARC3_PERSISTENT_FUNCTIONS_IMPORTS": "1",

    # ---- controls
    "EXPOSE_UNDO": "on",
}
if USE_PRIORITY_SCHEDULING:
    setup_env.update({
        "ARC3_MAX_ACTIVE_STREAMS": 10,
        "ARC3_PRIORITY_REFRESH_QUEUE": "1",
        "ARC3_PRIORITY_PACE": "0",
        "ARC3_PRIORITY_TAIL_FADE": "1",
        "ARC3_PRIORITY_TAIL_FADE_FRACTION": "0.2",
        "ARC3_PRIORITY_TAIL_LOOKUP": "remaining",
        "ARC3_PRIORITY_SCORE_NORMALIZATION": "1",
        "ARC3_PRIORITY_TOKEN_SCALE": 250000,     # 2026-10-15: was 62000 (code default); completion hazard is flat after ~40k tokens on a level, scheduler replay +0.8 (25 games) / +1.1 (110 games), scheduler replay study, 2026-10-15
    })
os.environ.update({key: str(value) for key, value in setup_env.items()})
_observations = WORKING_DIR / "observations"
if _observations.exists() and any(_observations.iterdir()):
    raise RuntimeError("A previous observation archive exists; use a fresh session/output directory.")
print(f"{len(setup_env)} runtime settings applied.", flush=True)

## Install runtime packages

In [ ]:
# arc3-step: install
if RUNTIME_ENVIRONMENT == "kaggle":
    # Warm the page cache: wheels now; model weights in the background once the server launches.
    arc3nb.precache(WHEELHOUSE_DIR)
    precache_model_thread = threading.Thread(target=arc3nb.precache, args=(MODEL_DIR, DRAFT_MODEL_DIR),
                                             kwargs={"delay": 60, "threads": 3}, daemon=True)
arc3nb.install_arc_runtime(RUNTIME_PATHS["competition"])
if RUNTIME_ENVIRONMENT == "colab":
    arc3nb.install_harness_dependencies(BUNDLE_DIR, RUNTIME_PATHS["competition"], WHEELHOUSE_DIR,
                                        RUNTIME_PATHS["allow_harness_pypi"])
_pth = arc3nb.expose_sources(BUNDLE_DIR)

## Self-test and provenance

A CPU self-test checks that the prompt contract and the sandbox agree. The run manifest records the
harness version and file digests, the settings, and the exact system prompt.

In [ ]:
# arc3-step: provenance
import platform
import shutil

from inference.agent.selftest import self_test

CPU_RESULT = self_test() if RUN_CPU_SELF_TESTS else {"status": "SKIPPED"}
print("Harness CPU self-test:", json.dumps(CPU_RESULT), flush=True)
if CPU_RESULT.get("status") not in {"PASS", "SKIPPED"}:
    raise RuntimeError("Harness self-test failed; the run was not started.")
_manifest = {
    "notebook": "arc-agi-3-final-solution", "runtime": RUNTIME_ENVIRONMENT, "submission": TRUE_SUBMISSION,
    "python": platform.python_version(), "run_stage": RUN_STAGE, "pilot_game_prefixes": PILOT_GAME_PREFIXES,
    "public_runtime_minutes": PUBLIC_RUNTIME_MINUTES, "public_excluded_games": PUBLIC_EXCLUDED_GAMES,
    "harness": HARNESS_INSTALL, "cpu_checks": CPU_RESULT, "runtime_settings": setup_env,
    "public_readiness": {"status": "NOT_APPLICABLE" if TRUE_SUBMISSION else "PENDING"},
}
_manifest.update(arc3nb.write_prompt_artifacts(WORKING_DIR))
shutil.copytree(BUNDLE_DIR / "src/ARC3-Inference/inference", WORKING_DIR / "artifacts/harness_sources/inference",
                ignore=shutil.ignore_patterns("__pycache__", "*.pyc"), dirs_exist_ok=True)
shutil.copy2(HARNESS_DIR / "HARNESS_MANIFEST.json", WORKING_DIR / "artifacts/HARNESS_MANIFEST.json")
arc3nb.save_json(WORKING_DIR / "experiment_manifest.json", _manifest)
print("System prompt:", _manifest["effective_system_prompt_chars"], "chars, sha256",
      _manifest["effective_system_prompt_sha256"][:12], flush=True)

## Model server

Kaggle: installs SGLang from the Pennyroyal wheelhouse and launches the server in the background
(`notebook/kaggle_sglang_launcher.py`). Colab: the adapter loads the model before this cell runs.

In [ ]:
# arc3-step: model
if RUNTIME_ENVIRONMENT == "kaggle":
    _launcher = NOTEBOOK_SUPPORT_DIR / "kaggle_sglang_launcher.py"
    exec(compile(_launcher.read_text(encoding="utf-8"), str(_launcher), "exec"), globals())
else:
    CFG = dict(RUNTIME_PATHS["model_settings"])
    with urlopen(f"http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/health", timeout=5) as _response:
        if _response.status != 200:
            raise RuntimeError("Model health check failed.")
    print("READY ->", SERVER_BASE_URL, flush=True)
_manifest["serving_settings"] = dict(CFG)
# The effort setting is a chat-template variable: confirm this model's template reads it, or the value is ignored silently.
_effort = os.environ.get("ARC3_REASONING_EFFORT", "").strip()
if _effort:
    _template = Path(MODEL_DIR) / "chat_template.jinja"
    _template_text = _template.read_text(encoding="utf-8") if _template.is_file() else ""
    _manifest["reasoning_effort"] = {"requested": _effort, "template_mentions_it": "reasoning_effort" in _template_text}
    if "reasoning_effort" not in _template_text:
        raise RuntimeError(f"ARC3_REASONING_EFFORT={_effort!r} but {_template} does not mention reasoning_effort; the server would ignore it.")
    print(f"reasoning_effort={_effort} (template reads it: True)", flush=True)
arc3nb.save_json(WORKING_DIR / "experiment_manifest.json", _manifest)

## Benchmark configuration

Restores the TAAF deployment target and benchmark, then sets the per-game budget and concurrency.
Make one-off changes to `bm`, `bm.games` or `bm.solver` here.

In [ ]:
# arc3-step: benchmark
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as _file:
    target = pickle.load(_file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as _file:
    bm = pickle.load(_file)
bm.job_dir = WORKING_DIR

bm.solver.analyzer_timeout = 900.0
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = not TRUE_SUBMISSION
bm.n_passes = 1
bm.game_weights = None
if TRUE_SUBMISSION:
    bm.solver.concurrency = 120               # all 110 hidden games admitted; ARC3_MAX_ACTIVE_STREAMS decode at once
    bm.solver.max_runtime_s_per_game = 532 * 60
else:
    bm.solver.concurrency = len(PILOT_GAME_PREFIXES) if RUN_STAGE == "PILOT" else 120
    bm.solver.max_runtime_s_per_game = PUBLIC_RUNTIME_MINUTES * 60
    if float(getattr(target, "max_runtime_s", 0.0) or 0.0) != 32400.0:
        raise RuntimeError(f"Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.")
print("benchmark.solver:", bm.solver)
_manifest["solver_controls"] = {
    "concurrency": bm.solver.concurrency, "active_streams": setup_env.get("ARC3_MAX_ACTIVE_STREAMS"),
    "max_runtime_s_per_game": bm.solver.max_runtime_s_per_game, "analyzer_timeout": bm.solver.analyzer_timeout,
    "n_passes": bm.n_passes, "max_generated_tokens_per_game": getattr(bm.solver, "max_generated_tokens_per_game", None),
    "model_dir": str(MODEL_DIR), "draft_model_dir": str(DRAFT_MODEL_DIR),
}
arc3nb.save_json(WORKING_DIR / "experiment_manifest.json", _manifest)

## Run

Outside a real submission the run waits for the model first, so startup never consumes a game budget,
and plays the offline environments (pilot subset or all 25 public games). A real submission plays the
competition games served by the Kaggle gateway.

In [ ]:
# arc3-step: run
_census_paths = ("/", str(WORKING_DIR))
if not TRUE_SUBMISSION:
    _stop_monitor = arc3nb.start_monitor(2, paths=_census_paths)
    try:
        _manifest["public_readiness"] = arc3nb.wait_for_model(SERVER_BASE_URL, 30 * 60)
    except TimeoutError:
        _manifest["public_readiness"] = {"status": "FAILED", "comparison_eligible": False}
        arc3nb.save_json(WORKING_DIR / "experiment_manifest.json", _manifest)
        raise

(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from the environment.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")          # the gateway's fixed local key
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    arc3nb.wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = arc3nb.competition_games(os.environ["ARC_BASE_URL"])
else:
    bm.games = arc3nb.offline_games(
        Path(RUNTIME_PATHS["competition"]) / "environment_files", excluded=PUBLIC_EXCLUDED_GAMES,
        prefixes=PILOT_GAME_PREFIXES if RUN_STAGE == "PILOT" else None,
        expected_count=len(PILOT_GAME_PREFIXES) if RUN_STAGE == "PILOT" else (None if PUBLIC_EXCLUDED_GAMES else 25))

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    _budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if _budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=_budget - min(600.0, _budget / 2))
_manifest["actual_game_order"] = [str(getattr(g, "env_name", getattr(g, "game_id", "unknown"))) for g in bm.games]
_manifest["actual_game_count"] = len(bm.games)
if not TRUE_SUBMISSION:
    _requested_seconds = PUBLIC_RUNTIME_MINUTES * 60
    if soft_end is not None and (soft_end - datetime.now()).total_seconds() < _requested_seconds:
        _manifest["public_readiness"].update(status="INSUFFICIENT_JOB_BUDGET", comparison_eligible=False)
        arc3nb.save_json(WORKING_DIR / "experiment_manifest.json", _manifest)
        raise RuntimeError("Not enough job time for the full public evaluation after startup; benchmark NOT started.")
    _manifest["public_readiness"].update(comparison_eligible=True, game_budget_seconds=_requested_seconds,
                                         benchmark_launch_epoch=time.time())
arc3nb.save_json(WORKING_DIR / "experiment_manifest.json", _manifest)
print(f"Starting benchmark: {len(bm.games)} games.", flush=True)

await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)

if not TRUE_SUBMISSION:
    # An offline run is not scored, but Kaggle still expects a submission.parquet output.
    import pandas as pd
    pd.DataFrame([["1_0", "1", True, 1]], columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(
        WORKING_DIR / "submission.parquet", index=False)

In [ ]:
# arc3-step: diagnostics
from html import escape

from IPython.display import HTML, display

_diagnostics_html = WORKING_DIR / "diagnostics.html"
if _diagnostics_html.is_file():
    # An iframe keeps the report's styles out of the notebook.
    display(HTML(f'<iframe srcdoc="{escape(_diagnostics_html.read_text(), quote=True)}" '
                 'width="100%" height="900" style="border:0"></iframe>'))
else:
    print("No diagnostics.html (real submissions run with minimal diagnostics).")

## Export

Writes `run_summary.json` and packs manifest, transcripts, observations, artifacts and logs into
the run bundle. On Colab the adapter copies the bundle to the Drive results folder.

In [ ]:
# arc3-step: export
from IPython.display import FileLink, display

_summary = arc3nb.run_summary(WORKING_DIR)
_summary.update(run_stage=RUN_STAGE, harness_version=HARNESS_INFO["version"])
arc3nb.save_json(WORKING_DIR / "run_summary.json", _summary)
print("Observation totals:", json.dumps(_summary.get("totals"), indent=2))
print("Transcript audit:", json.dumps(_summary["transcript_audit"], indent=2))
_bundle_path = arc3nb.export_bundle(WORKING_DIR, ARC3_BUNDLE_NAME)
display(FileLink(str(_bundle_path)))

In [ ]:
# arc3-step: parent
if RUNTIME_ENVIRONMENT == "colab" and COLAB_DISCONNECT_AFTER_EXPORT and globals().get("_ARC3_EXPORT_COMPLETE"):
    from google.colab import runtime
    runtime.unassign()